# Database Operations

**Module:** 02 — Vector Databases

Create, upsert, update/delete, search, filter, paginate.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Create collections correctly
- Idempotent upserts
- Filter+paginate safely


## Create Collection

**Definition.** Allocate named space with dim/metric/index settings.

**Why it matters.** Wrong params → expensive reindex.

**How it works.** API create; record embedder in registry.

**Intuition.** Filing cabinet with fixed drawer size.

**Common pitfalls.**
- Metric mismatch

**When to use.** New env/model/schema break.


In [ ]:
from dataclasses import dataclass, field
@dataclass
class Coll:
    name:str; dim:int; points:dict=field(default_factory=dict)
class Client:
    def __init__(self): self.c={}
    def create(self,n,d):
        assert n not in self.c; self.c[n]=Coll(n,d); return {'ok':n}
client=Client(); print(client.create('faq',384))


In [ ]:
import json
print(json.dumps({'vectors':{'size':384,'distance':'Cosine'}},indent=2)); print('YOUR_API_KEY')


In [ ]:
def ensure(client,n,d):
    if n in client.c: assert client.c[n].dim==d; return 'exists'
    client.create(n,d); return 'created'
print(ensure(client,'faq',384))


### Try it yourself — Create Collection

1. Fail loud on metric mismatch.


## Insert / Upsert

**Definition.** Insert new; upsert replace-by-id.

**Why it matters.** Re-runnable pipelines need stable IDs.

**How it works.** Batch {id,vector,payload}.

**Intuition.** Put version of ID on shelf.

**Common pitfalls.**
- Random UUIDs each run

**When to use.** Prefer upsert in prod.


In [ ]:
import numpy as np
def upsert(client,n,ids,V,P):
    for i,v,p in zip(ids,V,P): client.c[n].points[i]={'v':np.array(v),'p':p}
    return len(ids)
print(upsert(client,'faq',['a','b'],np.eye(2,384),[{},{}]))


In [ ]:
def batched(it,n=100):
    b=[]
    for x in it:
        b.append(x)
        if len(b)>=n: yield b; b=[]
    if b: yield b
print(list(batched(range(10),4)))


In [ ]:
import json
print(json.dumps({'upserted_count':128},indent=2))


### Try it yourself — Insert / Upsert

1. Stable chunk IDs for renamable wiki paths.


## Update / Delete

**Definition.** Change vectors/payload; remove by id/filter.

**Why it matters.** Edits, ACL, GDPR.

**How it works.** Metadata in place; text→re-embed; deletes may tombstone.

**Intuition.** Catalog must match shelves.

**Common pitfalls.**
- Broad delete filters
- Text update without re-embed

**When to use.** Edits/offboarding.


In [ ]:
client.c['faq'].points['a']['p']={'lang':'es'}; print(client.c['faq'].points['a'])


In [ ]:
def delete(coll,ids):
    for i in ids: coll.points.pop(i,None)
print(delete(client.c['faq'],['b']))


In [ ]:
import json
print(json.dumps({'filter':{'must':[{'key':'tenant','match':{'value':'gone'}}]}},indent=2))


### Try it yourself — Update / Delete

1. Runbook for user deletion request.


## Search

**Definition.** Similarity query (+filter) → scored IDs.

**Why it matters.** Product read path.

**How it works.** vector, top_k, filter, payloads.

**Intuition.** Near meaning among allowed rows.

**Common pitfalls.**
- tiny top_k before rerank

**When to use.** Every RAG retrieve.


In [ ]:
import numpy as np
def search(coll,q,k=3):
    q=np.array(q,float); q/=np.linalg.norm(q)+1e-9; sc=[]
    for pid,row in coll.points.items():
        v=row['v']/(np.linalg.norm(row['v'])+1e-9); sc.append((float(v@q),pid))
    return sorted(sc, reverse=True)[:k]
print(search(client.c['faq'],[1]+[0]*383))


In [ ]:
import json
print(json.dumps({'limit':5,'with_payload':True},indent=2))


In [ ]:
print([r for r in [(0.91,'1'),(0.2,'2')] if r[0]>=0.4])


### Try it yourself — Search

1. Define top_k + threshold for support bot.


## Metadata Filtering

**Definition.** Restrict by payload predicates.

**Why it matters.** Security/freshness/lang.

**How it works.** Boolean trees; bind tenant from auth.

**Intuition.** Similarity proposes; filters dispose.

**Common pitfalls.**
- Client tenant IDs
- Over-selective empty results

**When to use.** Multi-tenant corpora.


In [ ]:
print({'must':[{'key':'tenant','match':{'value':'acme'}}]})


In [ ]:
def backoff(primary,relax):
    a=[primary]
    for f in relax: a.append({k:v for k,v in primary.items() if k!=f})
    return a
print(backoff({'tenant':'acme','lang':'en','ver':'v2'},['ver','lang']))


In [ ]:
def overfetch(k,sel,s=2.0): return int(k/max(sel,1e-6)*s)
print(overfetch(10,0.05))


### Try it yourself — Metadata Filtering

1. Backoff that never crosses tenants.


## Pagination

**Definition.** Windows beyond page 1.

**Why it matters.** UI/export need stable iteration.

**How it works.** Prefer cursors; ANN deep pages hard.

**Intuition.** Bookmark, don't re-roll dice.

**Common pitfalls.**
- offset duplicates under mutation

**When to use.** UI/export; RAG rarely deep pages.


In [ ]:
ranked=[f'd{i}' for i in range(30)]
print(ranked[0:10], ranked[10:20])


In [ ]:
def nxt(ranked,cur,size):
    s=0 if cur is None else ranked.index(cur)+1; w=ranked[s:s+size]; return w, w[-1] if w else None
print(nxt(ranked,None,5))


In [ ]:
import json
print(json.dumps({'limit':10,'offset':10},indent=2))


### Try it yourself — Pagination

1. Why RAG ≠ deep pagination.


## Glossary

- **upsert**: Insert or replace by id
- **cursor**: Pagination token


## Summary & Key Takeaways

- Collections are contracts.
- Stable-ID upserts.
- Cursors > naive offsets.

### Practice

Mini client: create/upsert/search/delete + tenant guard.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
